# 01 — SAS: Single Agent System

```
User Question
     │
     ▼
Enterprise Data Analyst (single agent)
     │
     ├─ Step 1: Create execution plan (LLM #1)
     ├─ Step 2: Generate SQL query    (LLM #2)
     ├─ Step 3: Execute SQL            (Spark)
     └─ Step 4: Interpret results      (LLM #3)
     │
     ▼
Final Answer
```

## Properties
- Single agent, 3 LLM calls per question
- Full semantic layer injected directly into every prompt (no retrieval)
- Abstention via CANNOT_ANSWER if the question is unanswerable
- Serves as the baseline architecture for comparison

In [0]:
import time
import re
import uuid
from datetime import datetime
import pandas as pd
# ============================================================
# CONFIGURATION
# ============================================================
ARCHITECTURE = "SAS"

# LLM client — shared from mt_config (429-resilient, max_retries=5)
client = LLM_CLIENT

MODEL_ENDPOINT = MT_MODEL_ENDPOINT
AGENT_PROFILE = ANALYST_PROFILE

# Build full context (SAS = direct semantic delivery, full context in prompt)
context_dict = build_unified_context(ARCHITECTURE)
context_str = format_unified_context_for_prompt(context_dict, include_semantic=True)

print(f"SAS configured | {MODEL_ENDPOINT} | context: {len(context_str):,} chars")

In [0]:
# ============================================================
# SAS WORKFLOW — Single Agent, Direct Semantic Context
# ============================================================
# Step 1: PLAN     — Create execution plan (LLM #1)
# Step 2: SQL      — Generate SQL query    (LLM #2)
# Step 3: EXECUTE  — Run SQL on Spark
# Step 4: INTERPRET — Business answer       (LLM #3)
# ============================================================

def execute_sql_query(sql: str) -> str:
    """Execute SQL and return results as formatted string."""
    try:
        result_df = spark.sql(sql).toPandas()
        n_rows = len(result_df)
        if n_rows > 30:
            return f"({n_rows} rows total, showing first 30)\n" + result_df.head(30).to_string(index=False)
        return result_df.to_string(index=False)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"


def extract_sql_from_response(text: str) -> str:
    """Extract SQL query from LLM response text."""
    match = re.search(r"```(?:sql)?\s*(.+?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    lines = text.strip().split("\n")
    sql_lines = []
    capture = False
    for line in lines:
        stripped = line.strip().upper()
        if stripped.startswith(("SELECT", "WITH")):
            capture = True
        if capture:
            sql_lines.append(line)
    if sql_lines:
        return "\n".join(sql_lines).strip()
    if any(kw in text.upper() for kw in ["SELECT", "FROM"]):
        return text.strip()
    return None


def run_sas(question_dict: dict) -> dict:
    """
    Execute the SAS workflow for one question.
    Returns dict with answer, plan, sql, metrics, latency, tokens.
    """
    start_time = time.time()
    total_prompt_tokens = 0
    total_completion_tokens = 0
    llm_calls = 0
    steps_log = []

    question = question_dict["question"]

    try:
        # --- Step 1: PLAN ---
        step_start = time.time()
        plan_prompt = f"""{AGENT_PROFILE}

AVAILABLE CONTEXT:
{context_str}

QUESTION: {question}

Create a concise execution plan to answer this question using SQL.
List the tables needed, any filters/aggregations, and the approach.
If the question cannot be answered with available data, respond with CANNOT_ANSWER."""

        plan_resp = client.chat.completions.create(
            model=MODEL_ENDPOINT,
            messages=[{"role": "user", "content": plan_prompt}],
            temperature=LLM_PARAMS["plan"]["temperature"],
            max_tokens=LLM_PARAMS["plan"]["max_tokens"],
            **_seed_kwargs()
        )
        plan = plan_resp.choices[0].message.content
        total_prompt_tokens += plan_resp.usage.prompt_tokens
        total_completion_tokens += plan_resp.usage.completion_tokens
        llm_calls += 1
        steps_log.append({"step": 1, "agent": "Enterprise Data Analyst", "action": "plan", "status": "success", "latency_ms": (time.time() - step_start) * 1000})

        # Check for abstention
        if "CANNOT_ANSWER" in plan.upper():
            latency = time.time() - start_time
            return {
                "answer": plan, "plan": plan, "sql_generated": None,
                "sql_results": None, "sql_execution_status": "not_applicable",
                "latency_seconds": latency, "prompt_tokens": total_prompt_tokens,
                "completion_tokens": total_completion_tokens,
                "total_tokens": total_prompt_tokens + total_completion_tokens,
                "llm_calls": llm_calls, "steps_log": steps_log,
                "success": True, "error": None
            }

        # --- Step 2: SQL ---
        step_start = time.time()
        sql_prompt = f"""{AGENT_PROFILE}

AVAILABLE CONTEXT:
{context_str}

QUESTION: {question}
PLAN: {plan}

SQL RULES:
{chr(10).join('- ' + r for r in SQL_SAFETY_RULES)}

Generate a single SQL query to answer this question. Return ONLY the SQL inside ```sql``` blocks."""

        sql_resp = client.chat.completions.create(
            model=MODEL_ENDPOINT,
            messages=[{"role": "user", "content": sql_prompt}],
            temperature=LLM_PARAMS["sql"]["temperature"],
            max_tokens=LLM_PARAMS["sql"]["max_tokens"],
            **_seed_kwargs()
        )
        sql_text = sql_resp.choices[0].message.content
        total_prompt_tokens += sql_resp.usage.prompt_tokens
        total_completion_tokens += sql_resp.usage.completion_tokens
        llm_calls += 1

        sql_query = extract_sql_from_response(sql_text)
        steps_log.append({"step": 2, "agent": "Enterprise Data Analyst", "action": "generate_sql", "status": "success" if sql_query else "failed", "latency_ms": (time.time() - step_start) * 1000})

        # --- Step 3: EXECUTE ---
        sql_results = None
        sql_execution_status = "failed"
        if sql_query:
            step_start = time.time()
            sql_results = execute_sql_query(sql_query)
            sql_execution_status = "failed" if sql_results.startswith("SQL ERROR") else "success"
            steps_log.append({"step": 3, "agent": "Enterprise Data Analyst", "action": "execute_sql", "status": sql_execution_status, "latency_ms": (time.time() - step_start) * 1000})

        # --- Step 4: INTERPRET ---
        step_start = time.time()
        interpret_prompt = f"""{AGENT_PROFILE}

QUESTION: {question}
PLAN: {plan}
SQL QUERY: {sql_query or 'None generated'}
SQL RESULTS: {sql_results or 'No results'}

Provide a concise business answer based on the SQL results.
Cite specific numbers from the results. Do not invent values not present in the data.
If results are empty or errored, explain honestly what happened."""

        interp_resp = client.chat.completions.create(
            model=MODEL_ENDPOINT,
            messages=[{"role": "user", "content": interpret_prompt}],
            temperature=LLM_PARAMS["interpret"]["temperature"],
            max_tokens=LLM_PARAMS["interpret"]["max_tokens"],
            **_seed_kwargs()
        )
        answer = interp_resp.choices[0].message.content
        total_prompt_tokens += interp_resp.usage.prompt_tokens
        total_completion_tokens += interp_resp.usage.completion_tokens
        llm_calls += 1
        steps_log.append({"step": 4, "agent": "Enterprise Data Analyst", "action": "interpret", "status": "success", "latency_ms": (time.time() - step_start) * 1000})

        latency = time.time() - start_time
        return {
            "answer": answer, "plan": plan, "sql_generated": sql_query,
            "sql_results": str(sql_results)[:500] if sql_results else None,
            "sql_execution_status": sql_execution_status,
            "latency_seconds": latency, "prompt_tokens": total_prompt_tokens,
            "completion_tokens": total_completion_tokens,
            "total_tokens": total_prompt_tokens + total_completion_tokens,
            "llm_calls": llm_calls, "steps_log": steps_log,
            "success": True, "error": None
        }

    except Exception as e:
        latency = time.time() - start_time
        return {
            "answer": None, "plan": None, "sql_generated": None,
            "sql_results": None, "sql_execution_status": "failed",
            "latency_seconds": latency, "prompt_tokens": total_prompt_tokens,
            "completion_tokens": total_completion_tokens,
            "total_tokens": total_prompt_tokens + total_completion_tokens,
            "llm_calls": llm_calls, "steps_log": steps_log,
            "success": False, "error": str(e)
        }

print("\u2713 run_sas() defined | Workflow: Plan \u2192 SQL \u2192 Execute \u2192 Interpret (3 LLM calls)")

In [0]:
# ============================================================
# RUN SAS ARCHITECTURE
# ============================================================

# Respect outer scope (orchestrator sets this); default to all if standalone
if 'QUESTION_FILTER' not in dir():
    QUESTION_FILTER = None
run_questions = [q for q in EVALUATION_QUESTIONS if QUESTION_FILTER is None or q["id"] in QUESTION_FILTER]

print(f"SAS: {len(run_questions)} questions | direct semantic | {MODEL_ENDPOINT}")

all_results = []

for q in run_questions:
    run_id = str(uuid.uuid4())

    try:
        # Execute
        result = run_sas(q)

        # Compute evaluation metrics
        sql_exec_status = result.get("sql_execution_status", "failed")
        metrics = compute_all_metrics(
            run={"answer": result["answer"], "sql_generated": result.get("sql_generated"),
                 "sql_results": result.get("sql_results"), "success": result["success"],
                 "error": result.get("error")},
            question={"expected_answer": q["expected_answer"], "tables_needed": q.get("tables_needed", []),
                      "answerability_label": q.get("answerability_label", "answerable"),
                      "expected_claims": q.get("expected_claims", []),
                      "question": q["question"]},
            mode="SAS"
        )

        # Compact per-question output
        print(f"  {q['id']}: {result['latency_seconds']:.1f}s | {result['total_tokens']} tok | SQL:{sql_exec_status}")

        # Store
        all_results.append({
            "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
            "question_type": q["question_type"], "difficulty": q["difficulty"],
            "expected_answer": q["expected_answer"],
            "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
            "semantic_delivery": "direct", "profile_name": "Enterprise Data Analyst",
            "model": MODEL_ENDPOINT, "plan": result.get("plan"),
            "sql_generated": result.get("sql_generated"),
            "sql_results": result.get("sql_results"),
            "sql_execution_status": sql_exec_status,
            "generated_answer": result.get("answer"),
            "latency_seconds": result["latency_seconds"],
            "prompt_tokens": result.get("prompt_tokens", 0),
            "completion_tokens": result.get("completion_tokens", 0),
            "total_tokens": result["total_tokens"],
            "number_of_model_calls": result["llm_calls"],
            "llm_calls": result["llm_calls"],
            # Thesis KPIs (from question metadata)
            "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
            "sql_required": bool(q.get("tables_needed")),
            "sql_retry_count": result.get("sql_attempts", 1) - 1,  # retries = attempts - 1
            # Workflow
            "success": result["success"], "error": result.get("error"),
            **metrics,
        })

    except Exception as _loop_err:
        print(f"  {q['id']}: ✗ FAILED — {type(_loop_err).__name__}: {_loop_err}")
        all_results.append({
            "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
            "question_type": q["question_type"], "difficulty": q["difficulty"],
            "expected_answer": q["expected_answer"],
            "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
            "semantic_delivery": "direct", "profile_name": "Enterprise Data Analyst",
            "model": MODEL_ENDPOINT, "plan": None,
            "sql_generated": None, "sql_results": None,
            "sql_execution_status": "failed",
            "generated_answer": None,  # No answer produced — error in "error" field
            "latency_seconds": 0.0,
            "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
            "number_of_model_calls": 0, "llm_calls": 0,
            "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
            "sql_required": bool(q.get("tables_needed")),
            "sql_retry_count": 0,
            "success": False, "error": f"{type(_loop_err).__name__}: {_loop_err}",
        })

results_df = pd.DataFrame(all_results)

print(f"\u2713 SAS complete: {len(results_df)} runs | Avg latency: {results_df['latency_seconds'].mean():.1f}s")

In [0]:
# ============================================================
# CAPTURE SAS RESULTS + PERSIST TO DELTA
# ============================================================
sas_results_df = None

if "SAS" in ARCHITECTURES_TO_RUN and 'results_df' in dir() and len(results_df) > 0:
    sas_results_df = results_df.copy()
    all_experiment_results.extend(results_df.to_dict('records'))
    print(f"✓ SAS captured: {len(sas_results_df)} runs")
else:
    print("⚠ SAS not run or no results")

# --- Persist to Delta (overwrite — first architecture) ---
try:
    if sas_results_df is not None:
        _sdf = spark.createDataFrame(sas_results_df.astype(str))
        _sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(_RESULTS_TABLE)
        print(f"✓ SAS persisted to {_RESULTS_TABLE} (overwrite — first architecture)")
except Exception as _e:
    print(f"⚠ Delta persist failed: {_e}")

import gc; gc.collect()
_print_cumulative_comparison()